In [ ]:
import os
from pathlib import Path

def _find_project_root():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Could not find project root (.git not found)")

os.chdir(_find_project_root())

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset
from sklearn.datasets import fetch_openml

# --- 1. The Standard LeNet-5 Architecture ---
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.relu3 = nn.ReLU()
        self.fc2   = nn.Linear(120, 84)
        self.relu4 = nn.ReLU()
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x, get_features=True):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.relu4(self.fc2(x))
        if get_features: return x  # 84-dim representation for clustering
        return self.fc3(x)         # 10-dim logits for training

# --- 2. Robust Weight Loading / Training ---
def get_trained_model():
    model = LeNet5()
    # Attempting a different verified URL
    url = "https://github.com/vinhkhuc/PyTorch-Mini-Tutorials/raw/master/5_convolutional_net/lenet5_mnist.pth"
    
    try:
        print("Attempting to download pre-trained weights...")
        state_dict = torch.hub.load_state_dict_from_url(url, map_location='cpu', check_hash=False)
        model.load_state_dict(state_dict, strict=False)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Download failed (Error: {e}). Switching to Quick Train (approx 1 min)...")
        train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, 
                                               transform=transforms.Compose([transforms.ToTensor(), 
                                               transforms.Normalize((0.1307,), (0.3081,))])), 
                                 batch_size=64, shuffle=True)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data, get_features=False)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 200 == 0:
                print(f"Training Progress: [{batch_idx * len(data)}/60000]")
        print("Training complete.")
    
    model.eval()
    return model

# --- 3. Processing the Full Datasets ---

def save_full_mnist_lenet(filename="datasets/real/mnist_lenet_full.txt"):
    print("\n--- Processing MNIST ---")
    model = get_trained_model()
    
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    full_set = ConcatDataset([
        datasets.MNIST(root='./data', train=True, download=True, transform=transform),
        datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    ])
    loader = DataLoader(full_set, batch_size=2000, shuffle=False)

    all_repr = []
    with torch.no_grad():
        for images, _ in loader:
            all_repr.append(model(images).numpy())
    
    final_data = np.vstack(all_repr)
    np.savetxt(filename, final_data, fmt='%.8f')
    print(f"Saved MNIST: {final_data.shape} to {filename}")

def save_full_letter_recognition(filename="datasets/real/letter_recognition_full.txt"):
    print("Processing full Letter Recognition (20,000 samples)...")
    letter_data = fetch_openml(name='letter', version=1, as_frame=True)
    X = letter_data.data.values.astype(float)
    np.savetxt(filename, X, fmt='%.0f')
    print(f"Saved Letter Recognition: {X.shape} to {filename}")

def save_full_gas_turbine(filename="datasets/real/gas_turbine_full.txt"):
    print("\n--- Processing Gas Turbine ---")
    
    # Path to check for manual download
    local_zip = "gas_turbine_data.zip" 
    
    try:
        print("Fetching dataset via official UCI API (ID: 551)...")
        # fetch dataset 
        gas_turbine_emissions = fetch_ucirepo(id=551) 
        
        # Combine features and targets into one matrix for clustering
        X = gas_turbine_emissions.data.features
        y = gas_turbine_emissions.data.targets
        full_df = pd.concat([X, y], axis=1)
        
        data = full_df.values.astype(float)
        np.savetxt(filename, data, fmt='%.4f')
        print(f"Successfully saved Gas Turbine: {data.shape} to {filename}")

    except Exception as e:
        print(f"API Fetch failed: {e}")
        print("-" * 30)
        print("PLAN B: Manual Processing")
        print("1. Download the ZIP manually from: https://archive.ics.uci.edu/dataset/551/gas+turbine+co+and+nox+emissions+data+set")
        print(f"2. Save the ZIP file as '{local_zip}' in this same folder.")
        
        if os.path.exists(local_zip):
            import zipfile, io
            print(f"Found {local_zip}! Processing locally...")
            with zipfile.ZipFile(local_zip, 'r') as z:
                all_dfs = [pd.read_csv(z.open(f)) for f in z.namelist() if f.endswith('.csv')]
                data = pd.concat(all_dfs, ignore_index=True).values.astype(float)
                np.savetxt(filename, data, fmt='%.4f')
                print(f"Successfully saved Gas Turbine (Local): {data.shape}")
        else:
            print("Local ZIP not found. Please download it manually to continue.")


In [4]:
save_full_mnist_lenet()


--- Processing MNIST ---
Attempting to download pre-trained weights...
Downloading: "https://github.com/vinhkhuc/PyTorch-Mini-Tutorials/raw/master/5_convolutional_net/lenet5_mnist.pth" to /home/t3humphr/.cache/torch/hub/checkpoints/lenet5_mnist.pth
Download failed (Error: HTTP Error 404: Not Found). Switching to Quick Train (approx 1 min)...


100%|██████████| 9.91M/9.91M [00:00<00:00, 47.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 2.18MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 13.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.50MB/s]


Training Progress: [0/60000]
Training Progress: [12800/60000]
Training Progress: [25600/60000]
Training Progress: [38400/60000]
Training Progress: [51200/60000]
Training complete.
Saved MNIST: (70000, 84) to datasets/real/mnist_lenet_full.txt


In [11]:
save_full_letter_recognition()
save_full_gas_turbine()

Processing full Letter Recognition (20,000 samples)...


/home/t3humphr/miniconda3/envs/new-augpe2/lib/python3.9/site-packages/sklearn/datasets/_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


Saved Letter Recognition: (20000, 16) to datasets/real/letter_recognition_full.txt

--- Processing Gas Turbine ---
Fetching dataset via official UCI API (ID: 551)...
Successfully saved Gas Turbine: (36733, 12) to datasets/real/gas_turbine_full.txt


In [2]:
from sklearn.datasets import make_blobs
import os 
import numpy as np
PATH = "datasets/sklearn/"
os.makedirs(PATH, exist_ok=True)
for dimension in [4, 16, 64, 128]:
    for k in [4, 16, 64]:
        dataset, _ =  make_blobs(n_samples=20000, n_features=dimension, centers=k, random_state=1)
        filename = PATH+ f"sklearn_{k}_{dimension}.txt"
        np.savetxt(filename, dataset, fmt='%.4f')
print("Saved Datasets")  
  

Saved Datasets


In [1]:

# Extract ground truth labels for MNIST and Letter Recognition.

import numpy as np
from torchvision import datasets, transforms
from sklearn.datasets import fetch_openml
import warnings

# --- MNIST labels ---
# Original data was ConcatDataset([train, test]) with shuffle=False,
# so labels are simply train targets followed by test targets in dataset order.
print("Extracting MNIST labels...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_set = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
mnist_labels = np.concatenate([train_set.targets.numpy(), test_set.targets.numpy()])
np.savetxt("datasets/real/mnist_lenet_labels.txt", mnist_labels, fmt='%d')
print(f"Saved {mnist_labels.shape[0]} MNIST labels (0-9) -> datasets/real/mnist_lenet_labels.txt")

# --- Letter Recognition labels ---
# Original data used fetch_openml with no shuffling; labels are letter_data.target in the same row order.
print("\nExtracting Letter Recognition labels...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    letter_data = fetch_openml(name='letter', version=1, as_frame=True)
letter_labels_raw = letter_data.target.values          # strings: 'A', 'B', ..., 'Z'
unique_letters = sorted(np.unique(letter_labels_raw))  # alphabetical order
letter_to_int  = {l: i for i, l in enumerate(unique_letters)}
letter_labels  = np.array([letter_to_int[l] for l in letter_labels_raw])
np.savetxt("datasets/real/letter_recognition_labels.txt", letter_labels, fmt='%d')
print(f"Saved {letter_labels.shape[0]} Letter Recognition labels (0=A ... 25=Z) -> datasets/real/letter_recognition_labels.txt")
print(f"Mapping: {letter_to_int}")



Extracting MNIST labels...


100%|██████████| 9.91M/9.91M [00:00<00:00, 48.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 11.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.03MB/s]


Saved 70000 MNIST labels (0-9) -> datasets/real/mnist_lenet_labels.txt

Extracting Letter Recognition labels...
Saved 20000 Letter Recognition labels (0=A ... 25=Z) -> datasets/real/letter_recognition_labels.txt
Mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25}
